# Olist data exploration

A first look at five of the Olist e-commerce tables: what is in them, how they connect, and what is odd. Every column is read as text, so nothing gets converted behind our back.

<img src="https://i.imgur.com/HRhd2Y0.png" alt="How the Olist tables relate" width="1000">

*Table relationships, from the dataset's Kaggle page. This notebook looks at five of its nine tables.*

## 1. Load

Zip prefixes keep their leading zeros and blanks stay blanks.

In [1]:
from decimal import Decimal
from pathlib import Path

import pandas as pd

DATA = Path("../data")
FILES = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
}
t = {name: pd.read_csv(DATA / f, dtype=str, keep_default_na=False) for name, f in FILES.items()}
orders, items, payments, customers, products = t.values()

## 2. Size

How big is each table?

In [2]:
pd.DataFrame({k: v.shape for k, v in t.items()}, index=["rows", "columns"]).T

,rows,columns
orders,99441,8
items,112650,7
payments,103886,5
customers,99441,5
products,32951,9


Orders and customers have the same number of rows. Items and payments have more, so some orders have several of each.

## 3. Keys

What identifies a row in each table, and is it really unique?

In [3]:
keys = {
    "orders": (orders, ["order_id"]),
    "items": (items, ["order_id", "order_item_id"]),
    "payments": (payments, ["order_id", "payment_sequential"]),
    "customers": (customers, ["customer_id"]),
    "products": (products, ["product_id"]),
}
pd.DataFrame(
    {
        name: {
            "rows": len(df),
            "duplicate_keys": df.duplicated(cols).sum(),
            "duplicate_rows": df.duplicated().sum(),
        }
        for name, (df, cols) in keys.items()
    }
).T

,rows,duplicate_keys,duplicate_rows
orders,99441,0,0
items,112650,0,0
payments,103886,0,0
customers,99441,0,0
products,32951,0,0


Every key is unique and no row appears twice.

## 4. Blanks and bad values

Which columns have blanks, and do the numbers and dates parse?

In [4]:
# columns that have blanks
blanks = pd.concat({k: v.apply(lambda c: c.str.strip().eq("").sum()) for k, v in t.items()})
blanks[blanks > 0]

orders    order_approved_at                 160
          order_delivered_carrier_date     1783
          order_delivered_customer_date    2965
products  product_category_name             610
          product_name_lenght               610
          product_description_lenght        610
          product_photos_qty                610
          product_weight_g                    2
          product_length_cm                   2
          product_height_cm                   2
          product_width_cm                    2
dtype: int64

Three order dates have blanks (not every order is approved or delivered). 610 products have no category, and the same 610 count shows up for name length, description length and photo count. Two products lack weight and size.

In [5]:
# does "delivered" always come with a delivery date?
done = orders[orders.order_status == "delivered"]
pd.Series(
    {
        "delivered orders": len(done),
        "without delivery date": done.order_delivered_customer_date.str.strip().eq("").sum(),
    }
)

delivered orders         96478
without delivery date        8
dtype: int64

8 delivered orders have no delivery date.

In [6]:
def num(s):
    return pd.to_numeric(s, errors="coerce")


def ts(s):
    return pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S", errors="coerce")


def unparsable(col, parse):
    filled = col[col.str.strip() != ""]
    return int(parse(filled).isna().sum())


pd.Series(
    {
        "items.price": unparsable(items.price, num),
        "items.freight_value": unparsable(items.freight_value, num),
        "payments.payment_value": unparsable(payments.payment_value, num),
        "orders.order_purchase_timestamp": unparsable(orders.order_purchase_timestamp, ts),
    },
    name="values that fail to parse",
)

items.price                        0
items.freight_value                0
payments.payment_value             0
orders.order_purchase_timestamp    0
Name: values that fail to parse, dtype: int64

In [7]:
for s in (orders.order_status, payments.payment_type):
    print(s.value_counts().to_string(), "\n")

states = customers.customer_state
invalid_states = (~states.str.fullmatch(r"[A-Z]{2}")).sum()
zip_lengths = customers.customer_zip_code_prefix.str.len().value_counts().to_dict()
print("distinct states:", states.nunique(), "| not 2 uppercase letters:", invalid_states)
print("zip prefix lengths:", zip_lengths)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2 

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3 

distinct states: 27 | not 2 uppercase letters: 0
zip prefix lengths: {5: 99441}


Prices, payment values and purchase dates are never blank and all of them parse. There are 8 order statuses (most are delivered), 5 payment types (3 payments are `not_defined`), 27 valid states, and zip prefixes are always 5 characters.

## 5. How the tables connect

Does every row point to a parent, and how many children does an order have?

In [8]:
links = {
    "orders -> customers": (orders.customer_id, customers.customer_id),
    "items -> orders": (items.order_id, orders.order_id),
    "items -> products": (items.product_id, products.product_id),
    "payments -> orders": (payments.order_id, orders.order_id),
    "orders without items": (orders.order_id, items.order_id),
    "orders without payments": (orders.order_id, payments.order_id),
    "customers without orders": (customers.customer_id, orders.customer_id),
}
pd.Series(
    {name: int((~child.isin(parent)).sum()) for name, (child, parent) in links.items()},
    name="rows without a match",
)

orders -> customers           0
items -> orders               0
items -> products             0
payments -> orders            0
orders without items        775
orders without payments       1
customers without orders      0
Name: rows without a match, dtype: int64

In [9]:
status = orders.set_index("order_id").order_status
pd.DataFrame(
    {
        "no items": status[~status.index.isin(items.order_id)].value_counts(),
        "no payments": status[~status.index.isin(payments.order_id)].value_counts(),
    }
).fillna(0).astype(int)

,no items,no payments
order_status,,
canceled,164,0
created,5,0
delivered,0,1
invoiced,2,0
shipped,1,0
unavailable,603,0


No row points to a missing parent. 775 orders have no items, almost all unavailable (603) or canceled (164), and none are delivered. One delivered order has no payment.

In [10]:
# how many rows each order has (5 means "5 or more")
pd.DataFrame(
    {
        "orders by item count": items.groupby("order_id").size().clip(upper=5).value_counts(),
        "orders by payment count": payments.groupby("order_id").size().clip(upper=5).value_counts(),
    }
).sort_index()

,orders by item count,orders by payment count
1,88863,96479
2,7516,2382
3,1322,301
4,505,108
5,460,170


Most orders have one item and one payment, but 9,803 of the 98,666 orders with items (9.9%) have several items, and 2,961 of the 99,440 orders with payments (3.0%) have several payments. Joining items to payments directly would multiply those rows.

## 6. Sales, categories and payments

When do the orders happen, what is missing from the products, and does what customers paid match what they bought? Money is added up as `Decimal` so sums are exact.

In [11]:
# orders per purchase month
ts(orders.order_purchase_timestamp).dt.to_period("M").value_counts().sort_index()

order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64

Orders run from 2016-09 to 2018-10. 2016-09, 2016-10, 2016-12, 2018-09 and 2018-10 have only a handful (4, 324, 1, 16 and 4); every month from 2017-01 to 2018-08 has hundreds to thousands.

In [12]:
# delivered items with their category
delivered = orders.loc[orders.order_status == "delivered", ["order_id"]]
sold = items.merge(delivered, on="order_id").merge(
    products[["product_id", "product_category_name"]], on="product_id", how="left"
)
price = sold.price.map(Decimal)
no_category = sold.product_category_name.fillna("").str.strip().eq("")
cats = products.product_category_name
pd.Series(
    {
        "delivered items": len(sold),
        "items without category": no_category.sum(),
        "sales value": sum(price),
        "sales value without category": sum(price[no_category]),
        "distinct categories": cats.nunique(),
        "distinct after strip+lower": cats.str.strip().str.lower().nunique(),
    }
)

delivered items                      110197
items without category                 1537
sales value                     13221498.11
sales value without category      170726.63
distinct categories                      74
distinct after strip+lower               74
dtype: object

1,537 of 110,197 delivered items have no category: R$ 170,726.63 of R$ 13,221,498.11 in sales (1.3%). There are 74 categories and no case or spacing variants.

In [13]:
pd.Series(
    {
        "items price <= 0": (num(items.price) <= 0).sum(),
        "items freight < 0": (num(items.freight_value) < 0).sum(),
        "payments value < 0": (num(payments.payment_value) < 0).sum(),
        "payments value = 0": (num(payments.payment_value) == 0).sum(),
    }
)

items price <= 0      0
items freight < 0     0
payments value < 0    0
payments value = 0    9
dtype: int64

No price is zero or negative. 9 payments are exactly zero.

In [14]:
# paid total vs items + freight, per order
item_value = items.price.map(Decimal) + items.freight_value.map(Decimal)
item_total = item_value.groupby(items.order_id).sum()
paid_total = payments.payment_value.map(Decimal).groupby(payments.order_id).sum()
both = pd.concat([item_total.rename("items"), paid_total.rename("paid")], axis=1, join="inner")
gap = (both.paid - both["items"]).map(abs)
pd.Series(
    {
        "orders with items and payments": len(both),
        "gap = 0.00": (gap == 0).sum(),
        "0.00 < gap <= 0.01": ((gap > 0) & (gap <= Decimal("0.01"))).sum(),
        "0.01 < gap <= 0.02": ((gap > Decimal("0.01")) & (gap <= Decimal("0.02"))).sum(),
        "gap > 0.02": (gap > Decimal("0.02")).sum(),
    }
)

orders with items and payments    98665
gap = 0.00                        98089
0.00 < gap <= 0.01                  273
0.01 < gap <= 0.02                   36
gap > 0.02                          267
dtype: int64

In [15]:
# orders without items: do they still have payments?
no_items = paid_total[~paid_total.index.isin(items.order_id)]
pd.Series(
    {
        "orders without items, with payments": len(no_items),
        "of which paid > 0": (no_items > 0).sum(),
    }
)

orders without items, with payments    775
of which paid > 0                      772
dtype: int64

Of 98,665 orders with items and payments, 98,089 match exactly, 273 are off by 0.01 or less, 36 by 0.01 to 0.02 and 267 by more than 0.02. The 775 orders without items still have payments, and 772 of them paid more than zero.

## 7. Things that look off

Who pays more or less than the items cost, whether payment rows go missing, and where status and dates disagree.

In [16]:
# positive gap: the customer paid more than items + freight
signed = both.paid - both["items"]
big = signed[signed.abs() > Decimal("0.02")]
installments = payments.assign(n=payments.payment_installments.astype(int))
installments = installments.groupby("order_id").n.max()
pd.Series(
    {
        "orders with gap > 0.02": len(big),
        "paid more": (big > 0).sum(),
        "paid less": (big < 0).sum(),
        "paid more, in installments": (installments[big[big > 0].index] > 1).sum(),
        "median gap": sorted(big.abs())[len(big) // 2],
        "largest gap": big.abs().max(),
    }
)

orders with gap > 0.02           267
paid more                        244
paid less                         23
paid more, in installments       236
median gap                      7.04
largest gap                   182.81
dtype: object

The larger gaps mostly go one way: customers paid more than the items and freight cost. Nearly all of those used installments, which fits interest being added, though nothing in the data says so directly.

In [17]:
def sequence_gaps(df, seq):
    n = df.assign(s=df[seq].astype(int)).groupby("order_id").s.agg(["min", "max", "count"])
    return int(((n["min"] != 1) | (n["max"] != n["count"])).sum())


pd.Series(
    {
        "orders with payment number gaps": sequence_gaps(payments, "payment_sequential"),
        "orders with item number gaps": sequence_gaps(items, "order_item_id"),
    }
)

orders with payment number gaps    80
orders with item number gaps        0
dtype: int64

80 orders have payment numbers that skip or don't start at 1, which hints at missing payment rows. Item numbers have no such gaps.

In [18]:
zero = payments[payments.payment_value.map(Decimal) == 0]
print(zero.payment_type.value_counts().to_string(), "\n")
print(status.loc[zero.order_id].value_counts().to_string(), "\n")

canceled = orders[orders.order_status == "canceled"]
print("canceled orders with a delivery date:", canceled.order_delivered_customer_date.ne("").sum())
print("payments with 0 installments:", (payments.payment_installments == "0").sum())

payment_type
voucher        6
not_defined    3 

order_status
delivered    4
canceled     3
shipped      2 

canceled orders with a delivery date: 6
payments with 0 installments: 2


The zero-value payments are vouchers (6) or `not_defined` (3), spread over delivered, canceled and shipped orders. Status and dates don't always agree either: 6 canceled orders have a delivery date.

## 8. What to remember

| What | Rows | Note |
|---|---|---|
| Orders without items | 775 (772 paid more than zero) | Mostly unavailable (603) or canceled (164). None are delivered. |
| Delivered order without payment | 1 | The only order with no payment. |
| Paid total differs from items + freight by more than 0.01 | 303 | 36 differ by 0.01 to 0.02, 267 by more. Of those 267, 244 paid more than the items cost. |
| Paid total differs by 0.01 or less, but not zero | 273 | Within a cent of the items total. |
| Orders with several items / several payments | 9,803 / 2,961 | Joining the two tables directly multiplies rows. |
| Orders with gaps in payment numbers | 80 | Possibly missing payment rows. |
| Delivered items without category | 1,537 (R$ 170,726.63, 1.3% of sales) | Their products have a blank category. |
| Payments of exactly zero | 9 | 6 vouchers and 3 `not_defined`. |
| Delivered orders without delivery date | 8 | Status says delivered, the date is blank. |
| Canceled orders with a delivery date | 6 | |
| Months with very few orders | 2016-09 to 2016-12, 2018-09 to 2018-10 | 2017-01 to 2018-08 have hundreds to thousands each. |
| Misspelled columns | n/a | `product_name_lenght`, `product_description_lenght`. |
